In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)
import evaluate

In [3]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Torch:", torch.__version__)

Device: cpu
Torch: 2.10.0+cpu


In [4]:
liar_train = pd.read_csv("../data/liar/liar_train_clean.csv")
liar_valid = pd.read_csv("../data/liar/liar_valid_clean.csv")
liar_test  = pd.read_csv("../data/liar/liar_test_clean.csv")

isot = pd.read_csv("../data/isot/isot_clean.csv")

# Safety: ensure correct types
for df in [liar_train, liar_valid, liar_test, isot]:
    df["statement"] = df["statement"].fillna("").astype(str)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

print("LIAR train:", liar_train.shape, "ISOT:", isot.shape)
print("LIAR label counts:\n", liar_train["label"].value_counts())
print("ISOT label counts:\n", isot["label"].value_counts())


LIAR train: (10240, 2) ISOT: (44898, 2)
LIAR label counts:
 label
1    6591
0    3649
Name: count, dtype: int64
ISOT label counts:
 label
0    23481
1    21417
Name: count, dtype: int64


In [5]:
ISOT_SUBSET_TOTAL = 4000   # changed to 4000 cuz CPU is too slow
PER_CLASS = ISOT_SUBSET_TOTAL // 2

isot_0 = isot[isot["label"] == 0].sample(n=PER_CLASS, random_state=SEED)
isot_1 = isot[isot["label"] == 1].sample(n=PER_CLASS, random_state=SEED)

isot_subset = pd.concat([isot_0, isot_1], axis=0).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("ISOT subset shape:", isot_subset.shape)
print(isot_subset["label"].value_counts())

ISOT subset shape: (4000, 2)
label
0    2000
1    2000
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

isot_train, isot_temp = train_test_split(
    isot_subset, test_size=0.3, random_state=SEED, stratify=isot_subset["label"]
)
isot_valid, isot_test = train_test_split(
    isot_temp, test_size=0.5, random_state=SEED, stratify=isot_temp["label"]
)

print("ISOT subset splits:", isot_train.shape, isot_valid.shape, isot_test.shape)

ISOT subset splits: (2800, 2) (600, 2) (600, 2)


In [7]:
def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[["statement", "label"]].reset_index(drop=True))

liar_train_ds = to_hf_dataset(liar_train)
liar_valid_ds = to_hf_dataset(liar_valid)
liar_test_ds  = to_hf_dataset(liar_test)

isot_train_ds = to_hf_dataset(isot_train)
isot_valid_ds = to_hf_dataset(isot_valid)
isot_test_ds  = to_hf_dataset(isot_test)

print(liar_train_ds, isot_train_ds)

Dataset({
    features: ['statement', 'label'],
    num_rows: 10240
}) Dataset({
    features: ['statement', 'label'],
    num_rows: 2800
})


In [8]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    }

In [9]:
os.makedirs("results/metrics", exist_ok=True)
os.makedirs("results/models", exist_ok=True)

# CPU-friendly settings
MAX_LEN_LIAR = 128     # short statements
MAX_LEN_ISOT = 128     # articles (subset) - keep moderate for CPU

BATCH_SIZE = 8
EPOCHS = 2             # On CPU, 2 epochs for ISOT can be brutal. One epoch is enough for comparative study + cross-domain analysis.
# 2 is realistic on CPU, can increase to 3 if time allows

print("Config:", {"BATCH_SIZE": BATCH_SIZE, "EPOCHS": EPOCHS, "MAX_LEN_LIAR": MAX_LEN_LIAR, "MAX_LEN_ISOT": MAX_LEN_ISOT})

Config: {'BATCH_SIZE': 8, 'EPOCHS': 2, 'MAX_LEN_LIAR': 128, 'MAX_LEN_ISOT': 128}


In [10]:
def train_and_evaluate(
    model_name: str,
    train_ds: Dataset,
    valid_ds: Dataset,          # kept for signature consistency (not used in training)
    test_in_domain: Dataset,
    test_cross_domain: Dataset,
    max_len: int,
    run_tag: str
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["statement"], truncation=True, max_length=max_len)

    train_tok = train_ds.map(tokenize, batched=True)
    test_in_tok = test_in_domain.map(tokenize, batched=True)
    test_cross_tok = test_cross_domain.map(tokenize, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    out_dir = f"results/models/{run_tag}_{model_name.replace('/', '_')}"
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="no",              # ✅ no in-training eval (faster, less hanging)
        save_strategy="no",              # ✅ no checkpoints (less Windows I/O freezing)
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,              # ✅ fast on CPU
        weight_decay=0.01,
        logging_steps=25,
        dataloader_num_workers=0,        # ✅ important for Windows stability
        dataloader_pin_memory=False,     # ✅ CPU only
        report_to="none",
        load_best_model_at_end=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    in_metrics = trainer.evaluate(test_in_tok)
    cross_metrics = trainer.evaluate(test_cross_tok)

    return {
        "model": model_name,
        "run_tag": run_tag,
        "in_domain": in_metrics,
        "cross_domain": cross_metrics
    }

In [11]:
# Run BERT & RoBERTa (LIAR-trained)
results = []

# LIAR-trained: in-domain LIAR test, cross-domain ISOT test
bert_liar = train_and_evaluate(
    model_name="bert-base-uncased",
    train_ds=liar_train_ds,
    valid_ds=liar_valid_ds,
    test_in_domain=liar_test_ds,
    test_cross_domain=isot_test_ds,
    max_len=MAX_LEN_LIAR,
    run_tag="TRAIN_LIAR"
)
results.append(bert_liar)

roberta_liar = train_and_evaluate(
    model_name="roberta-base",
    train_ds=liar_train_ds,
    valid_ds=liar_valid_ds,
    test_in_domain=liar_test_ds,
    test_cross_domain=isot_test_ds,
    max_len=MAX_LEN_LIAR,
    run_tag="TRAIN_LIAR"
)
results.append(roberta_liar)

print("Done: LIAR-trained transformer experiments.")

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\emmab\AppData\Local\Temp\ipykernel_34076\3228773896.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
25,0.663300
50,0.667100
75,0.690900
100,0.655000
125,0.678200
150,0.660000
175,0.654000
200,0.648100
225,0.660700
250,0.639200


...

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\emmab\AppData\Local\Temp\ipykernel_34076\3228773896.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
25,0.645200
50,0.678700
75,0.681600
100,0.645100
125,0.689700
150,0.655700
175,0.681400
200,0.646400
225,0.676700
250,0.621200


Done: LIAR-trained transformer experiments.


In [12]:
# Run BERT & RoBERTa (ISOT-trained)
# ISOT-trained: in-domain ISOT test, cross-domain LIAR test
bert_isot = train_and_evaluate(
    model_name="bert-base-uncased",
    train_ds=isot_train_ds,
    valid_ds=isot_valid_ds,
    test_in_domain=isot_test_ds,
    test_cross_domain=liar_test_ds, 
    max_len=MAX_LEN_ISOT,
    run_tag="TRAIN_ISOT"
)
results.append(bert_isot)

roberta_isot = train_and_evaluate(
    model_name="roberta-base",
    train_ds=isot_train_ds,
    valid_ds=isot_valid_ds,
    test_in_domain=isot_test_ds,
    test_cross_domain=liar_test_ds,
    max_len=MAX_LEN_ISOT,
    run_tag="TRAIN_ISOT"
)
results.append(roberta_isot)

print("Done: ISOT-trained transformer experiments.")

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\emmab\AppData\Local\Temp\ipykernel_34076\3228773896.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
25,0.412900
50,0.042700
75,0.034600
100,0.025200
125,0.002100
150,0.001600
175,0.001200
200,0.001100
225,0.001100
250,0.000900


...

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\emmab\AppData\Local\Temp\ipykernel_34076\3228773896.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
25,0.605200
50,0.054900
75,0.038000
100,0.037000
125,0.000900
150,0.000700
175,0.000500
200,0.000500
225,0.052900
250,0.000700


Done: ISOT-trained transformer experiments.


In [13]:
# Converting results into a table
def extract_metrics(r, kind):
    m = r[kind]
    return {
        "accuracy": m.get("eval_accuracy"),
        "precision": m.get("eval_precision"),
        "recall": m.get("eval_recall"),
        "f1": m.get("eval_f1")
    }

rows = []
for r in results:
    rows.append({
        "Model": r["model"],
        "TrainingDomain": r["run_tag"],
        "TestType": "In-domain",
        **extract_metrics(r, "in_domain")
    })
    rows.append({
        "Model": r["model"],
        "TrainingDomain": r["run_tag"],
        "TestType": "Cross-domain",
        **extract_metrics(r, "cross_domain")
    })

transformers_df = pd.DataFrame(rows)
transformers_df

,Model,TrainingDomain,TestType,accuracy,precision,recall,f1
0,bert-base-uncased,TRAIN_LIAR,In-domain,0.636148,0.636364,0.998759,0.777402
1,bert-base-uncased,TRAIN_LIAR,Cross-domain,0.501667,0.500835,1.000000,0.667408
2,roberta-base,TRAIN_LIAR,In-domain,0.636938,0.636867,0.998759,0.777778
3,roberta-base,TRAIN_LIAR,Cross-domain,0.500000,0.500000,1.000000,0.666667
4,bert-base-uncased,TRAIN_ISOT,In-domain,1.000000,1.000000,1.000000,1.000000
5,bert-base-uncased,TRAIN_ISOT,Cross-domain,0.363852,0.000000,0.000000,0.000000
6,roberta-base,TRAIN_ISOT,In-domain,1.000000,1.000000,1.000000,1.000000
7,roberta-base,TRAIN_ISOT,Cross-domain,0.363062,0.400000,0.002481,0.004932


In [14]:
out_path = "results/metrics/transformers_metrics.csv"
transformers_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: results/metrics/transformers_metrics.csv


In [15]:
# Compute the F1 drop
def f1_of(model, train_domain, test_type):
    row = transformers_df[
        (transformers_df["Model"] == model) &
        (transformers_df["TrainingDomain"] == train_domain) &
        (transformers_df["TestType"] == test_type)
    ].iloc[0]
    return float(row["f1"])

for model in ["bert-base-uncased", "roberta-base"]:
    # LIAR-trained drop
    f1_in = f1_of(model, "TRAIN_LIAR", "In-domain")
    f1_out = f1_of(model, "TRAIN_LIAR", "Cross-domain")
    print(f"{model} F1 drop (LIAR→LIAR vs LIAR→ISOT):", round(f1_in - f1_out, 4))

    # ISOT-trained drop
    f1_in = f1_of(model, "TRAIN_ISOT", "In-domain")
    f1_out = f1_of(model, "TRAIN_ISOT", "Cross-domain")
    print(f"{model} F1 drop (ISOT→ISOT vs ISOT→LIAR):", round(f1_in - f1_out, 4))

bert-base-uncased F1 drop (LIAR→LIAR vs LIAR→ISOT): 0.11
bert-base-uncased F1 drop (ISOT→ISOT vs ISOT→LIAR): 1.0
roberta-base F1 drop (LIAR→LIAR vs LIAR→ISOT): 0.1111
roberta-base F1 drop (ISOT→ISOT vs ISOT→LIAR): 0.9951
